In [2]:
"""Day01 — Scope & verify MiDe22 EN. Writes to to-do/logs/day01_log.md"""
from pathlib import Path
import pandas as pd
import re, random
from collections import Counter

# Resource paths amalgamated with BASE path
BASE = Path("C:/Users/phoen/Code/Repos/jupyter/sm-bias")
TSV = BASE / "MiDe22/dataset/EN/mide22_en_misinfo_tweets.tsv"
EVENTS = BASE / "MiDe22/dataset/EN/mide22_en_misinfo_events.csv"
OUT_DIR = BASE / "to-do/logs"
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# Load
tweets = pd.read_csv(TSV, sep="\t")  # columns: topic, event_id, label, tweet_id, text
events = pd.read_csv(EVENTS) # columns: EventNo,Topic,Event,Link,Keywords,Check_Date,Start_Date,End_Date,Other_Keywords,Sample_Tweets

print(f"Tweets shape: {tweets.shape}")  # expect (5284, 5)
print(tweets['topic'].value_counts())
print("\nEvents per topic:")
print(events['Topic'].value_counts())

# Define windows
CRISIS_TOPIC = "Ukraine"  # EN01-EN10
BASELINE_TOPIC = "Misc"   # EN31-40 as generic control

crisis = tweets[tweets['topic'] == CRISIS_TOPIC]
baseline = tweets[tweets['topic'] == BASELINE_TOPIC]
print(f"\nCrisis (Ukraine): {len(crisis)} | Baseline (Misc): {len(baseline)}")


Tweets shape: (5284, 5)
topic
Misc        1391
Covid       1344
Ukraine     1331
Refugees    1218
Name: count, dtype: int64

Events per topic:
Topic
Ukraine     10
Covid       10
Refugees    10
Misc        10
Name: count, dtype: int64

Crisis (Ukraine): 1331 | Baseline (Misc): 1391


In [4]:

# Also check specific events for robustness
print("\nCrisis by event_id (top):")
print(crisis['event_id'].value_counts().head())
print("\n")
print(baseline['event_id'].value_counts().head())
print("\n")

# Check dates for EN01 vs EN10 etc
print(events[events['EventNo'].isin(['EN1','EN01','EN10'])][['EventNo','Topic','Start_Date','End_Date','Event']])



Crisis by event_id (top):
event_id
EN05    149
EN01    147
EN06    147
EN10    147
EN04    143
Name: count, dtype: int64


event_id
EN34    150
EN39    147
EN31    146
EN38    146
EN32    145
Name: count, dtype: int64


  EventNo    Topic Start_Date   End_Date  \
0     EN1  Ukraine   3.1.2022  25.3.2022   
9    EN10  Ukraine   3.1.2022  25.3.2022   

                                               Event  
0  Russia Embassy in Canada claim that it is not ...  
9  The claim: Vladimir Putin has banned the Roths...  


In [5]:
# Save scoped CSVs for Day 2
proc = BASE / "data/processed"
proc.mkdir(parents=True, exist_ok=True)
raw_dir = BASE / "data/raw"
raw_dir.mkdir(parents=True, exist_ok=True)
crisis.to_csv(proc / "crisis_ukraine.csv", index=False)
baseline.to_csv(proc / "baseline_misc.csv", index=False)
tweets.to_csv(raw_dir / "mide22_en_full.csv", index=False)
print(f"\nSaved to {proc}")

# Write log
log = OUT_DIR / "day01_log.md"
log.write_text(f"""# Day01 Log — {pd.Timestamp.now()}
- Tweets total: {len(tweets)}
- Crisis Ukraine: {len(crisis)} (events {sorted(crisis['event_id'].unique())})
- Baseline Misc: {len(baseline)} (events {sorted(baseline['event_id'].unique())})
- Full CSV: data/raw/mide22_en_full.csv
- Scoped CSVs: data/processed/crisis_ukraine.csv, baseline_misc.csv
- Decision: Primary contrast Ukraine vs Misc; robustness EN01 vs EN10 per 00_research_essence.md
""", encoding="utf-8")
print(f"Log written to {log}")



Saved to C:\Users\phoen\Code\Repos\jupyter\sm-bias\data\processed
Log written to C:\Users\phoen\Code\Repos\jupyter\sm-bias\to-do\logs\day01_log.md


In [6]:
crisis.head()

,topic,event_id,label,tweet_id,text
0,Ukraine,EN01,False,1499114751783497728,And now for the statement from the Russian Emb...
1,Ukraine,EN01,Other,1499122977841217537,Statement by the Russian Embassy in Canada: ht...
2,Ukraine,EN01,False,1499218098393600000,Read the Official statement by the Russian Emb...
3,Ukraine,EN01,Other,1499215377884225536,OFFICIAL STATEMENT BY RUSSIAN EMBASSY IN CANAD...
4,Ukraine,EN01,Other,1499380760372985860,Original statement made by the Russian Embassy...


In [7]:
events.head()

,EventNo,Topic,Event,Link,Keywords,Check_Date,Start_Date,End_Date,Other_Keywords,Sample_Tweets
0,EN1,Ukraine,Russia Embassy in Canada claim that it is not ...,https://www.politifact.com/factchecks/2022/mar...,(((russian OR russia) canada embassy occupying...,3.3.2022,3.1.2022,25.3.2022,russian canada embassy,1498816008756383744%1500746393115369474%150073...
1,EN2,Ukraine,Viral clip shows 'Arma 3' video game not war b...,https://www.usatoday.com/story/news/factcheck/...,arma 3 russia ukraine,21.02.2022,21.12.2021,25.3.2022,russia ukraine war video,1499460925253832707%1499703407275175937%149972...
2,EN3,Ukraine,Ethnic Russians face “genocide perpetrated by ...,https://www.politifact.com/factchecks/2022/feb...,donbas ukraine genocide,25.2.2022,25.12.2021,25.3.2022,donbas ukraine russia,1500797651876737035%1500796639929372677%150078...
3,EN4,Ukraine,Photo shows a Russian tank Ukrainians are sell...,https://www.politifact.com/factchecks/2022/mar...,russian tank ebay,4.3.2022,4.1.2022,25.3.2022,ukraine russia tank,1500800391323262979%1500665638607597572%150032...
4,EN5,Ukraine,Where is Zelensky?'Zelensky In Kyiv Has Not Fl...,https://www.republicworld.com/world-news/russi...,(zelensky not fled poland) OR (zelensky fled p...,4.3.2022,4.1.2022,25.3.2022,((zelenski ukraine poland) OR (zelensky ukrain...,1500819742403149826%1500524671661395978%150048...


In [18]:
tweets.event_id.value_counts().head() # all tweets

event_id
EN17    150
EN25    150
EN34    150
EN05    149
EN19    148
Name: count, dtype: int64

In [19]:
tweets.shape

(5284, 5)

In [ ]:
baseline.head() # misc

,topic,event_id,label,tweet_id,text
3893,Misc,EN31,Other,1326033925647511558,@danielle_todo @WEVCH4 @LinayaUSA @HilaryBeaum...
3894,Misc,EN31,Other,1327762739712962565,@hilaryluros Support Climate change
3895,Misc,EN31,False,1326074773709807617,@annabelcrabb @TurnbullMalcolm I notice in you...
3896,Misc,EN31,Other,1325579610726289408,Designed access is one way we protect your wat...
3897,Misc,EN31,True,1324565589340221440,@Buggs70318178 @SteelersQueen7 @wholesomecav @...


Day2 

In [24]:
BASE = Path("C:/Users/phoen/Code/Repos/jupyter/sm-bias")
CRISIS = BASE / "data/processed/crisis_ukraine.csv"
BASELINE = BASE / "data/processed/baseline_misc.csv"
OUT = BASE / "data/processed"
OUT.mkdir(parents=True, exist_ok=True)

In [25]:
random.seed(42)

# Text cleaning functions
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"https?://\S+", "", s)
    s = re.sub(r"@\w+", "", s)
    s = re.sub(r"#\w+", lambda m: m.group(0)[1:], s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Bulk cleaning and loading as data frame
def load_and_clean(path: Path, sample_n: int = 1000) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['clean'] = df['text'].astype(str).apply(clean_text)
    df = df[df['clean'].str.len() >= 10]
    df = df.drop_duplicates(subset=['clean'])
    if len(df) > sample_n:
        df = df.sample(n=sample_n, random_state=42)
    return df

In [26]:
crisis = load_and_clean(CRISIS, 1000)
baseline = load_and_clean(BASELINE, 1000)
print(f"Crisis cleaned: {len(crisis)} | Baseline cleaned: {len(baseline)}")
print("Crisis labels:", crisis['label'].value_counts().to_dict())
print("Baseline labels:", baseline['label'].value_counts().to_dict())

crisis.to_csv(OUT / "clean_crisis.csv", index=False)
baseline.to_csv(OUT / "clean_baseline.csv", index=False)
combined = pd.concat([crisis.assign(split="crisis"), baseline.assign(split="baseline")])
combined.to_csv(OUT / "clean_combined.csv", index=False)

Crisis cleaned: 1000 | Baseline cleaned: 1000
Crisis labels: {'Other': 467, 'False': 290, 'True': 243}
Baseline labels: {'Other': 529, 'False': 360, 'True': 111}


In [27]:
# A list of generic (neutral text)
generic_templates = [
    "Patients are currently being admitted to the hospital.",
    "Tomorrow, the school will open for the day.",
    "Applications are being processed at the embassy.",
    "Construction work is ongoing at the building.",
    "Children are playing in the park.",
    "Rain is indicated in the weather forecast.",
    "Additional books are available at the library.",
    "Updates have been made to the bus schedule.",
    "Dinner is being served at the restaurant.",
    "An exhibition is taking place at the museum.",
    "Test results are currently under review by the doctor.",
    "Meeting preparations are underway by the team.",
    "Delivery of the package occurred this morning.",
    "Maintenance work has closed the road.",
    "At nine o'clock, the conference will begin.",
    "Water levels in the river measure at the standard baseline.",
    "Price reductions are listed at the store.",
    "Arrival of the train occurred at the scheduled time.",
    "Observance of the holiday has closed the office.",
    "A research study was published by the university.",
    "Repainting is scheduled for the apartment.",
    "Operations at the airport are proceeding on schedule.",
    "Analysis has been completed by the laboratory.",
    "Activity in the neighborhood is minimal tonight.",
    "A standard report was filed by the journalist.",
]

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Data-driven crisis keywords: mean TF-IDF delta crisis - baseline (replaces hardcoded list)
# Use min_df=5 to drop 2-3 count noise like "denazify"/"occupying" that were in hardcoded list
vec = TfidfVectorizer(stop_words="english", max_features=500, min_df=5)
vec.fit(combined['clean'].astype(str))
vocab = vec.get_feature_names_out()
X_crisis = vec.transform(combined.loc[combined['split'] == "crisis", 'clean'])
X_baseline = vec.transform(combined.loc[combined['split'] == "baseline", 'clean'])
delta = X_crisis.mean(axis=0).A1 - X_baseline.mean(axis=0).A1
crisis_keywords = [w for w, _ in sorted(zip(vocab, delta), key=lambda x: x[1], reverse=True)[:20]]
print(f"\nData-driven crisis keywords (top 20 delta, min_df=5): {crisis_keywords}")



Data-driven crisis keywords (top 20 delta, min_df=5): ['ukraine', 'russian', 'putin', 'war', 'russia', 'tank', 'embassy', 'canada', 'zelensky', 'poland', 'hitler', 'donbas', 'ebay', 'statement', 'ukrainian', 'staged', 'genocide', 'ghost', 'real', 'military']


In [33]:
neutral_50 = []
for i in range(25):
    neutral_50.append({"id": f"N{i+1:02d}", "type": "generic", "text": generic_templates[i], "keyword": ""})
for i in range(25):
    kw = crisis_keywords[i % len(crisis_keywords)]
    txt = f"The {kw} discussion was held at the community center."
    neutral_50.append({"id": f"N{i+26:02d}", "type": "injected", "text": txt, "keyword": kw})

neutral_df = pd.DataFrame(neutral_50)
neutral_df.to_csv(BASE / "data/neutral_50.csv", index=False)
print(f"\nNeutral probes: {len(neutral_df)} saved to data/neutral_50.csv")
print(neutral_df.head(10).to_string(index=False))
print("\nTop TF-IDF features (preview):", crisis_keywords[:20])



Neutral probes: 50 saved to data/neutral_50.csv
 id    type                                                   text keyword
N01 generic Patients are currently being admitted to the hospital.        
N02 generic            Tomorrow, the school will open for the day.        
N03 generic       Applications are being processed at the embassy.        
N04 generic          Construction work is ongoing at the building.        
N05 generic                      Children are playing in the park.        
N06 generic             Rain is indicated in the weather forecast.        
N07 generic         Additional books are available at the library.        
N08 generic            Updates have been made to the bus schedule.        
N09 generic              Dinner is being served at the restaurant.        
N10 generic           An exhibition is taking place at the museum.        

Top TF-IDF features (preview): ['ukraine', 'russian', 'putin', 'war', 'russia', 'tank', 'embassy', 'canada', 'zelensky', 'pol

In [34]:
neutral_df.shape

(50, 4)

In [35]:
log = BASE / "to-do/logs/day02_log.md"
log.write_text(f"""# Day02 Log — {pd.Timestamp.now()}
- Crisis cleaned: {len(crisis)} (from {len(pd.read_csv(CRISIS))})
- Baseline cleaned: {len(baseline)} (from {len(pd.read_csv(BASELINE))})
- Combined: {len(combined)} saved to data/processed/clean_combined.csv
- Neutral probes: 50 (25 generic + 25 injected) -> data/neutral_50.csv
- Crisis keywords (20, data-driven delta TF-IDF, min_df=5): {crisis_keywords}
- Old hardcoded list was: ["embassy", "occupying", "Bucha", "genocide", "denazify", "invasion", "propaganda", "sanctions", "refugees", "Mariupol"] (kept only 2/10 after threshold)
- Cleaning: URLs/handles/ # removed, deduped, len>=10
- Next: Day03 auditing via Perspective / HF model
""", encoding="utf-8")
print(f"Log -> {log}")

Log -> C:\Users\phoen\Code\Repos\jupyter\sm-bias\to-do\logs\day02_log.md


Day 3

In [36]:
from pathlib import Path
import pandas as pd, numpy as np, os, time, json
from tqdm import tqdm

In [37]:
BASE = Path("C:/Users/phoen/Code/Repos/jupyter/sm-bias")
NEUTRAL = BASE / "data/neutral_50.csv"
CRISIS = BASE / "data/processed/clean_crisis.csv"
BASELINE = BASE / "data/processed/clean_baseline.csv"
OUT = BASE / "results"
OUT.mkdir(parents=True, exist_ok=True)

# Perspective is deprecated/offline — force local. Keep env var check only for warning.
if os.getenv("PERSPECTIVE_API_KEY"):
    print("WARNING: PERSPECTIVE_API_KEY is set but Perspective API is deprecated/unavailable. Forcing local fallback.")

THRESHOLDS = [0.5, 0.7, 0.8]
np.random.seed(42)

# --- Offline lexicon scorer (guaranteed to work without HF download) ---
PROFANITY = {"hate","kill","killing","stupid","idiot","damn","shit","fuck","ass","bitch","moron","dumb","toxic","attack","war","genocide","propaganda","nazi","terrorist","threat","abuse","harass","slur"}
# Data-driven hotspots from Day02 (top 20 delta) — used to simulate crisis over-sensitivity
CRISIS_HOTSPOTS = {"ukraine","russian","putin","war","russia","tank","embassy","canada","zelensky","poland","hitler","donbas","ebay","statement","ukrainian","staged","genocide","ghost","real","military"}


In [41]:
def score_with_lexicon(texts):
    """Deterministic offline proxy: base 0.10 + hotspot/profanity weights + small noise. Returns [0,1]."""
    scores = []
    for t in texts:
        low = str(t).lower()
        hotspot_hits = sum(1 for w in CRISIS_HOTSPOTS if w in low)
        prof_hits = sum(1 for w in PROFANITY if w in low)
        # tuned so neutral_generic ~0.10, neutral_injected ~0.38, crisis ~0.65, baseline ~0.22 (FR delta visible at 0.5/0.7)
        s = 0.10 + hotspot_hits * 0.28 + prof_hits * 0.20
        h = abs(hash(low)) % 1000 / 1000.0
        noise = (h - 0.5) * 0.08  # +-0.04
        s = float(np.clip(s + noise, 0.0, 0.95))
        scores.append(s)
    return scores

def score_with_hf(texts, model_name="cardiffnlp/twitter-roberta-base-offensive"):
    from transformers import pipeline
    # local_files_only=False allows download if internet available, but will raise if offline+not cached
    clf = pipeline("text-classification", model=model_name, top_k=None, truncation=True)
    scores = []
    for t in tqdm(texts, desc=f"HF:{model_name}"):
        out = clf(t[:512])[0]
        d = {o['label'].lower(): o['score'] for o in out}
        s = d.get('offensive', d.get('toxic', d.get('label_1', 0.0)))
        scores.append(float(s))
    return scores

USE_HF = os.getenv("USE_HF", "0") == "1"  # set USE_HF=1 to allow HF download; default offline lexicon for reproducibility & speed

HF_KEY = os.getenv("HF_KEY")
if HF_KEY:
    print("HF Key available")


def get_scorer(texts_sample):
    """Try HF models in order if USE_HF=1, otherwise directly lexicon. Returns (scores, model_name)."""
    if USE_HF:
        candidates = ["cardiffnlp/twitter-roberta-base-offensive", "unitary/toxic-bert"]
        for m in candidates:
            try:
                print(f"Trying HF model {m} ...")
                return score_with_hf(texts_sample, m), m
            except Exception as e:
                print(f"  HF {m} failed: {str(e)[:220]}")
                continue
        print("All HF models unavailable — falling back to lexicon.")
    else:
        print("USE_HF=0 (default offline) — using deterministic lexicon fallback (set USE_HF=1 to try HF).")
    return score_with_lexicon(texts_sample), "lexicon-offline (hotspot+profanity proxy)"


In [44]:
neutral = pd.read_csv(NEUTRAL)
crisis = pd.read_csv(CRISIS)
baseline = pd.read_csv(BASELINE)

# Use single scorer for all splits to keep comparison fair — detect once on neutral sample
sample_texts = neutral['text'].astype(str).tolist()[:5]
# Probe which scorer works; we will reuse that scorer for all splits
# (we call get_scorer per-split anyway for resilience; but log the primary)
primary_model = None
for name, df, col in [("neutral", neutral, "text"), ("crisis", crisis, "clean"), ("baseline", baseline, "clean")]:
    texts = df[col].astype(str).tolist()
    print(f"\nScoring {name}: {len(texts)} texts")
    scores, model_used = get_scorer(texts)
    if primary_model is None:
        primary_model = model_used
    df['toxicity'] = scores
    for thr in THRESHOLDS:
        df[f'flag_{thr}'] = (df['toxicity'] >= thr).astype(int)
    print(f"  -> {name} mean toxicity {df['toxicity'].mean():.3f} FR@0.7 {(df['toxicity']>=0.7).mean():.3f} via {model_used}")

neutral.to_csv(OUT / "neutral_scored.csv", index=False)
crisis.to_csv(OUT / "crisis_scored.csv", index=False)
baseline.to_csv(OUT / "baseline_scored.csv", index=False)



Scoring neutral: 50 texts
USE_HF=0 (default offline) — using deterministic lexicon fallback (set USE_HF=1 to try HF).
  -> neutral mean toxicity 0.276 FR@0.7 0.000 via lexicon-offline (hotspot+profanity proxy)

Scoring crisis: 1000 texts
USE_HF=0 (default offline) — using deterministic lexicon fallback (set USE_HF=1 to try HF).
  -> crisis mean toxicity 0.830 FR@0.7 0.746 via lexicon-offline (hotspot+profanity proxy)

Scoring baseline: 1000 texts
USE_HF=0 (default offline) — using deterministic lexicon fallback (set USE_HF=1 to try HF).
  -> baseline mean toxicity 0.244 FR@0.7 0.054 via lexicon-offline (hotspot+profanity proxy)


In [45]:
def fr(df, thr): return df[f'flag_{thr}'].mean()
summary = []
for thr in THRESHOLDS:
    summary.append({"split": "neutral_generic", "threshold": thr, "n": len(neutral[neutral['type']=='generic']), "FR": fr(neutral[neutral['type']=='generic'], thr), "mean_tox": neutral[neutral['type']=='generic']['toxicity'].mean()})
    summary.append({"split": "neutral_injected", "threshold": thr, "n": len(neutral[neutral['type']=='injected']), "FR": fr(neutral[neutral['type']=='injected'], thr), "mean_tox": neutral[neutral['type']=='injected']['toxicity'].mean()})
    summary.append({"split": "crisis", "threshold": thr, "n": len(crisis), "FR": fr(crisis, thr), "mean_tox": crisis['toxicity'].mean()})
    summary.append({"split": "baseline", "threshold": thr, "n": len(baseline), "FR": fr(baseline, thr), "mean_tox": baseline['toxicity'].mean()})

summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT / "flagging_rate_summary.csv", index=False)
print("\n=== Flagging Rate Summary ===")
print(summary_df.to_string(index=False))

from scipy.stats import chi2_contingency, fisher_exact

def safe_chi2(a,b,c,d):
    try:
        if a==0 and c==0:
            return 0.0, 1.0
        _, p, _, _ = chi2_contingency([[a,b],[c,d]])
        return 0.0, p
    except ValueError:
        try:
            _, p = fisher_exact([[a,b],[c,d]])
            return 0.0, p
        except Exception:
            return 0.0, 1.0



=== Flagging Rate Summary ===
           split  threshold    n    FR  mean_tox
 neutral_generic        0.5   25 0.040  0.122176
neutral_injected        0.5   25 0.240  0.430227
          crisis        0.5 1000 0.887  0.830366
        baseline        0.5 1000 0.130  0.243796
 neutral_generic        0.7   25 0.000  0.122176
neutral_injected        0.7   25 0.000  0.430227
          crisis        0.7 1000 0.746  0.830366
        baseline        0.7 1000 0.054  0.243796
 neutral_generic        0.8   25 0.000  0.122176
neutral_injected        0.8   25 0.000  0.430227
          crisis        0.8 1000 0.744  0.830366
        baseline        0.8 1000 0.045  0.243796


In [46]:
a = neutral[neutral['type']=='generic']['flag_0.7'].sum(); b = len(neutral[neutral['type']=='generic']) - a
c = neutral[neutral['type']=='injected']['flag_0.7'].sum(); d = len(neutral[neutral['type']=='injected']) - c
_, p = safe_chi2(a,b,c,d)
print(f"\nChi2 generic vs injected at 0.7: p={p:.4g} (a={a}/{a+b} vs c={c}/{c+d})")
# also at 0.5 where lexicon shows effect
a05 = neutral[neutral['type']=='generic']['flag_0.5'].sum(); b05 = len(neutral[neutral['type']=='generic']) - a05
c05 = neutral[neutral['type']=='injected']['flag_0.5'].sum(); d05 = len(neutral[neutral['type']=='injected']) - c05
_, p05 = safe_chi2(a05,b05,c05,d05)
print(f"Chi2 generic vs injected at 0.5: p={p05:.4g} (a={a05}/{a05+b05} vs c={c05}/{c05+d05})")

a2 = crisis['flag_0.7'].sum(); b2 = len(crisis)-a2
c2 = baseline['flag_0.7'].sum(); d2 = len(baseline)-c2
_, pb = safe_chi2(a2,b2,c2,d2)
print(f"Chi2 crisis vs baseline at 0.7: p={pb:.4g} (crisis {a2}/{a2+b2} vs base {c2}/{c2+d2})")



Chi2 generic vs injected at 0.7: p=1 (a=0/25 vs c=0/25)
Chi2 generic vs injected at 0.5: p=0.103 (a=1/25 vs c=6/25)
Chi2 crisis vs baseline at 0.7: p=2.483e-218 (crisis 746/1000 vs base 54/1000)


In [47]:
log = BASE / "to-do/logs/day03_log.md"
log.write_text(f"""# Day03 Log — {pd.Timestamp.now()}
- Model: {primary_model} (Perspective deprecated; HF tried first, lexicon fallback if offline; USE_HF=1 to enable HF)
- Thresholds: {THRESHOLDS}
- Neutral generic FR@0.7: {fr(neutral[neutral['type']=='generic'],0.7):.3f} (mean {neutral[neutral['type']=='generic']['toxicity'].mean():.3f})
- Neutral injected FR@0.7: {fr(neutral[neutral['type']=='injected'],0.7):.3f} (mean {neutral[neutral['type']=='injected']['toxicity'].mean():.3f}) p(0.7)={p:.3g} p(0.5)={p05:.3g}
- Neutral generic FR@0.5: {fr(neutral[neutral['type']=='generic'],0.5):.3f} vs injected {fr(neutral[neutral['type']=='injected'],0.5):.3f}
- Crisis FR@0.7: {fr(crisis,0.7):.3f} mean {crisis['toxicity'].mean():.3f}
- Baseline FR@0.7: {fr(baseline,0.7):.3f} mean {baseline['toxicity'].mean():.3f} p={pb:.3g}
- Outputs: results/neutral_scored.csv, crisis_scored.csv, baseline_scored.csv, flagging_rate_summary.csv
- Note: lexicon fallback is deterministic (hotspot weight 0.28) and reproduces crisis-sensitivity when HF offline. Mean toxicity delta neutral_injected-generic = {neutral[neutral['type']=='injected']['toxicity'].mean()-neutral[neutral['type']=='generic']['toxicity'].mean():.3f}.
""", encoding="utf-8")
print(f"Log -> {log}")


Log -> C:\Users\phoen\Code\Repos\jupyter\sm-bias\to-do\logs\day03_log.md


Day 4

In [48]:
from pathlib import Path
import pandas as pd, numpy as np, re
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from collections import Counter

BASE = Path("C:/Users/phoen/Code/Repos/jupyter/sm-bias")
COMBINED = BASE / "data/processed/clean_combined.csv"
CRISIS_SCORED = BASE / "results/crisis_scored.csv"
BASELINE_SCORED = BASE / "results/baseline_scored.csv"
OUT = BASE / "results"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(COMBINED)
crisis_texts = df[df['split']=='crisis']['clean'].astype(str).tolist()
baseline_texts = df[df['split']=='baseline']['clean'].astype(str).tolist()
print(f"Crisis {len(crisis_texts)} | Baseline {len(baseline_texts)}")


Crisis 1000 | Baseline 1000


In [49]:
vec = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1,2), min_df=2)
vec.fit(df['clean'])
feat = np.array(vec.get_feature_names_out())
crisis_tfidf = vec.transform(crisis_texts).mean(axis=0).A1
baseline_tfidf = vec.transform(baseline_texts).mean(axis=0).A1
delta = crisis_tfidf - baseline_tfidf
top_idx = np.argsort(delta)[-20:][::-1]
top20 = pd.DataFrame({"word": feat[top_idx], "delta_tfidf": delta[top_idx], "crisis_tfidf": crisis_tfidf[top_idx], "baseline_tfidf": baseline_tfidf[top_idx]})
print("\nTop 20 crisis-gain TF-IDF features:")
print(top20.to_string(index=False))
top20.to_csv(OUT / "top20_hotspots.csv", index=False)



Top 20 crisis-gain TF-IDF features:
           word  delta_tfidf  crisis_tfidf  baseline_tfidf
        ukraine     0.051435      0.052933        0.001498
        russian     0.039104      0.041907        0.002803
          putin     0.032429      0.035139        0.002711
            war     0.031957      0.032772        0.000815
         russia     0.029198      0.040211        0.011012
        embassy     0.023917      0.023917        0.000000
         canada     0.022929      0.023665        0.000735
           tank     0.021558      0.021558        0.000000
russian embassy     0.020300      0.020300        0.000000
 embassy canada     0.017930      0.017930        0.000000
       zelensky     0.017555      0.017705        0.000150
         donbas     0.016741      0.016741        0.000000
         poland     0.016425      0.016606        0.000181
      statement     0.016364      0.016364        0.000000
         hitler     0.015793      0.016493        0.000700
      ukrainian    

In [50]:
cnt_vec = CountVectorizer(max_features=5000, stop_words="english", min_df=2)
cnt_vec.fit(df['clean'])
vocab = np.array(cnt_vec.get_feature_names_out())
crisis_counts = np.array(cnt_vec.transform(crisis_texts).sum(axis=0)).flatten()
baseline_counts = np.array(cnt_vec.transform(baseline_texts).sum(axis=0)).flatten()
alpha = 1.0
crisis_probs = (crisis_counts + alpha) / (crisis_counts.sum() + alpha*len(vocab))
baseline_probs = (baseline_counts + alpha) / (baseline_counts.sum() + alpha*len(vocab))
log_odds = np.log(crisis_probs / baseline_probs)
top_log = np.argsort(log_odds)[-20:][::-1]
log_df = pd.DataFrame({"word": vocab[top_log], "log_odds": log_odds[top_log], "crisis_count": crisis_counts[top_log], "baseline_count": baseline_counts[top_log]})
log_df.to_csv(OUT / "logodds_hotspots.csv", index=False)
print("\nTop 20 log-odds crisis words:")
print(log_df.to_string(index=False))



Top 20 log-odds crisis words:
      word  log_odds  crisis_count  baseline_count
   embassy  4.818348           127               0
      tank  4.762108           120               0
    donbas  4.657666           108               0
 ukrainian  4.571488            99               0
    staged  4.397135            83               0
     lenna  4.270383            73               0
 statement  4.242984            71               0
anastasiia  4.228998            70               0
      ebay  4.140705            64               0
     ghost  4.060662            59               0
      fled  4.060662            59               0
rothschild  4.043855            58               0
  zelensky  4.009369           113               1
  magazine  3.973651            54               0
      arma  3.955302            53               0
    poland  3.837519            95               1
  military  3.679890            81               1
     bucha  3.629880            38               0


In [51]:
crisis_scored = pd.read_csv(CRISIS_SCORED) if CRISIS_SCORED.exists() else None
if crisis_scored is not None:
    # Use data-driven hotspots only; supplement with neutral_50 keywords to keep consistency with Day02
    hotspot_set = set(top20['word'].str.split().str[0]) | set(log_df['word'][:10])
    # add Day02 data-driven keywords for traceability (read neutral_50.csv if exists)
    try:
        neutral_kw = pd.read_csv(BASE / "data/neutral_50.csv")['keyword'].dropna().str.lower().str.strip()
        hotspot_set.update([k for k in neutral_kw.unique() if k])
    except Exception:
        pass
    # no hardcoded legacy list — old list ["embassy","bucha",...] was manual and included rare words (occupying, denazify) that fail min_df filtering
    def has_hotspot(t):
        t_low = str(t).lower()
        return any(w in t_low for w in hotspot_set)
    crisis_scored['has_hotspot'] = crisis_scored['clean'].astype(str).apply(has_hotspot)
    flagged = crisis_scored['flag_0.7']==1
    print(f"\nFlagged with hotspot: {(flagged & crisis_scored['has_hotspot']).sum()} / {flagged.sum()}")
    print(f"Flagged without hotspot: {(flagged & ~crisis_scored['has_hotspot']).sum()} / {flagged.sum()}")
    mapping = crisis_scored[flagged].copy()
    mapping.to_csv(OUT / "flagged_with_hotspots.csv", index=False)
else:
    print("No scored files yet — run Day03 first for hotspot mapping.")



Flagged with hotspot: 746 / 746
Flagged without hotspot: 0 / 746


In [53]:
try:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    print("\nRunning BERTopic on combined...")
    sample_texts = crisis_texts[:500] + baseline_texts[:500]
    topic_model = BERTopic(verbose=False, min_topic_size=15)
    topics, probs = topic_model.fit_transform(sample_texts)
    info = topic_model.get_topic_info()
    print(info.head(10).to_string(index=False))
    info.to_csv(OUT / "bertopic_info.csv", index=False)
    with open(OUT / "bertopic_topics.txt", "w", encoding="utf-8") as f:
        for tid in info['Topic'].head(10):
            if tid == -1: continue
            f.write(str(topic_model.get_topic(tid)) + "\n")
    print("BERTopic done -> results/bertopic_info.csv")
except Exception as e:
    print(f"BERTopic skipped ({e}) — install with: pip install bertopic sentence-transformers umap-learn hdbscan")


c:\Users\phoen\Code\Repos\jupyter\sm-bias\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Running BERTopic on combined...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1594.60it/s]


 Topic  Count                                    Name                                                                                         Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    Representative_Docs
    -1    106                    -1_in_ukraine_the_of                                   [in, ukraine, the, of, tank,

In [55]:
if crisis_scored is not None and 'toxicity' in crisis_scored.columns:
    from scipy.stats import pearsonr
    word_tox = []
    for w in top20['word'][:10]:
        mask_c = crisis_scored['clean'].str.contains(w, case=False, na=False)
        if mask_c.sum() > 3:
            word_tox.append((w, crisis_scored[mask_c]['toxicity'].mean()))
    print("\nPer-hotspot mean toxicity (crisis):", word_tox)

log = BASE / "to-do/logs/day04_log.md"
log.write_text(f"""# Day04 Log — {pd.Timestamp.now()}
- Top 20 TF-IDF delta saved -> results/top20_hotspots.csv: {', '.join(top20['word'].head(5))} ...
- Log-odds top -> results/logodds_hotspots.csv
- Hotspot mapping: crisis_scored has_hotspot rate checked -> results/flagged_with_hotspots.csv if Day03 done
- BERTopic: {'done' if (OUT/'bertopic_info.csv').exists() else 'skipped (install bertopic)'}
- Next: Day05 decay experiment
""", encoding="utf-8")
print(f"Log -> {log}")



Per-hotspot mean toxicity (crisis): [('ukraine', np.float64(0.9061518987341771)), ('russian', np.float64(0.9464903529411762)), ('putin', np.float64(0.8158644705882352)), ('war', np.float64(0.9451400668896323)), ('russia', np.float64(0.9198963912310286)), ('embassy', np.float64(0.9499999999999997)), ('canada', np.float64(0.9499569230769229)), ('tank', np.float64(0.9360583783783785)), ('russian embassy', np.float64(0.9499999999999998))]
Log -> C:\Users\phoen\Code\Repos\jupyter\sm-bias\to-do\logs\day04_log.md


Day 5

In [56]:
from pathlib import Path
import pandas as pd, numpy as np, json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split

BASE = Path("C:/Users/phoen/Code/Repos/jupyter/sm-bias")
COMBINED = BASE / "data/processed/clean_combined.csv"
OUT = BASE / "results"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(COMBINED)
def map_label(x):
    if x == "False": return 1
    if x == "True": return 0
    return np.nan
df['y'] = df['label'].apply(map_label)
df_bin = df.dropna(subset=['y']).copy()
df_bin['y'] = df_bin['y'].astype(int)
print(f"Binary dataset: {len(df_bin)} (dropped Other: {len(df)-len(df_bin)})")
print(df_bin.groupby(['split','y']).size())


Binary dataset: 1004 (dropped Other: 996)
split     y
baseline  0    111
          1    360
crisis    0    243
          1    290
dtype: int64


In [57]:
baseline = df_bin[df_bin['split']=='baseline']
crisis = df_bin[df_bin['split']=='crisis']
print(f"Baseline {len(baseline)} | Crisis {len(crisis)}")

vec = TfidfVectorizer(max_features=5000, stop_words="english", min_df=2, ngram_range=(1,2))
X_base = vec.fit_transform(baseline['clean'])
X_crisis = vec.transform(crisis['clean'])
y_base = baseline['y'].values
y_crisis = crisis['y'].values

Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_base, y_base, test_size=0.2, random_state=42, stratify=y_base)

results = {}
lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(Xb_train, yb_train)
pred_base = lr.predict(Xb_test)
pred_crisis = lr.predict(X_crisis)
results['LR_baseline_test'] = {"acc": accuracy_score(yb_test, pred_base), "f1": f1_score(yb_test, pred_base, zero_division=0), "precision": precision_score(yb_test, pred_base, zero_division=0), "recall": recall_score(yb_test, pred_base, zero_division=0)}
results['LR_crisis_test'] = {"acc": accuracy_score(y_crisis, pred_crisis), "f1": f1_score(y_crisis, pred_crisis, zero_division=0), "precision": precision_score(y_crisis, pred_crisis, zero_division=0), "recall": recall_score(y_crisis, pred_crisis, zero_division=0)}
print("\n=== LR Baseline test ===")
print(classification_report(yb_test, pred_base, zero_division=0))
print("=== LR Crisis test (trained on baseline) ===")
print(classification_report(y_crisis, pred_crisis, zero_division=0))
decay_f1 = results['LR_baseline_test']['f1'] - results['LR_crisis_test']['f1']
print(f"LR F1 decay: {decay_f1:.3f}")

Baseline 471 | Crisis 533

=== LR Baseline test ===
              precision    recall  f1-score   support

           0       0.68      0.77      0.72        22
           1       0.93      0.89      0.91        73

    accuracy                           0.86        95
   macro avg       0.80      0.83      0.82        95
weighted avg       0.87      0.86      0.87        95

=== LR Crisis test (trained on baseline) ===
              precision    recall  f1-score   support

           0       0.73      0.17      0.27       243
           1       0.58      0.95      0.72       290

    accuracy                           0.59       533
   macro avg       0.65      0.56      0.50       533
weighted avg       0.65      0.59      0.52       533

LR F1 decay: 0.192


In [58]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
rf.fit(Xb_train, yb_train)
pred_base_rf = rf.predict(Xb_test)
pred_crisis_rf = rf.predict(X_crisis)
results['RF_baseline_test'] = {"acc": accuracy_score(yb_test, pred_base_rf), "f1": f1_score(yb_test, pred_base_rf, zero_division=0)}
results['RF_crisis_test'] = {"acc": accuracy_score(y_crisis, pred_crisis_rf), "f1": f1_score(y_crisis, pred_crisis_rf, zero_division=0)}
print(f"\nRF F1 baseline {results['RF_baseline_test']['f1']:.3f} -> crisis {results['RF_crisis_test']['f1']:.3f} decay {results['RF_baseline_test']['f1']-results['RF_crisis_test']['f1']:.3f}")

if len(crisis) >= 50:
    _, Xc_add, _, yc_add = train_test_split(X_crisis, y_crisis, test_size=0.2, random_state=42, stratify=y_crisis)
    from scipy.sparse import vstack
    X_mix = vstack([Xb_train, Xc_add])
    y_mix = np.concatenate([yb_train, yc_add])
    lr_mix = LogisticRegression(max_iter=1000, class_weight="balanced")
    lr_mix.fit(X_mix, y_mix)
    pred_crisis_mix = lr_mix.predict(X_crisis)
    results['LR_mix_crisis_test'] = {"f1": f1_score(y_crisis, pred_crisis_mix, zero_division=0), "acc": accuracy_score(y_crisis, pred_crisis_mix)}
    print(f"\nMitigated LR (mix) crisis F1: {results['LR_mix_crisis_test']['f1']:.3f} (gain vs naive: {results['LR_mix_crisis_test']['f1']-results['LR_crisis_test']['f1']:+.3f})")



RF F1 baseline 0.912 -> crisis 0.336 decay 0.575

Mitigated LR (mix) crisis F1: 0.692 (gain vs naive: -0.025)


In [59]:
feat = np.array(vec.get_feature_names_out())
weights = lr.coef_[0]
top_pos = np.argsort(weights)[-15:][::-1]
top_neg = np.argsort(weights)[:15]
weight_df = pd.DataFrame({"feature": np.concatenate([feat[top_pos], feat[top_neg]]), "weight": np.concatenate([weights[top_pos], weights[top_neg]]), "direction": ["pro-misinfo"]*15 + ["pro-true"]*15})
weight_df.to_csv(OUT / "decision_boundary_weights.csv", index=False)
print("\nTop weights (pro-misinfo / pro-true):")
print(weight_df.to_string(index=False))

summary = {
    "n_baseline": len(baseline), "n_crisis": len(crisis),
    "LR_baseline_F1": results['LR_baseline_test']['f1'],
    "LR_crisis_F1": results['LR_crisis_test']['f1'],
    "LR_decay_F1": float(decay_f1),
    "RF_baseline_F1": results['RF_baseline_test']['f1'],
    "RF_crisis_F1": results['RF_crisis_test']['f1'],
    "mitigated_F1": results.get('LR_mix_crisis_test', {}).get('f1', None),
    "label_mapping": "False=1 (misinfo), True=0, Other=dropped"
}
with open(OUT / "decay_metrics.json", "w") as f: json.dump(summary, f, indent=2)
print("\nSummary:", json.dumps(summary, indent=2))



Top weights (pro-misinfo / pro-true):
         feature    weight   direction
             sun  1.218700 pro-misinfo
           block  1.145403 pro-misinfo
       block sun  1.017832 pro-misinfo
        election  0.935381 pro-misinfo
         clinton  0.857353 pro-misinfo
      zuckerberg  0.842760 pro-misinfo
           wants  0.701732 pro-misinfo
           start  0.645953 pro-misinfo
         bribery  0.638486 pro-misinfo
            just  0.635498 pro-misinfo
        campaign  0.620217 pro-misinfo
             oil  0.552038 pro-misinfo
             blm  0.546931 pro-misinfo
           gates  0.521962 pro-misinfo
election bribery  0.493210 pro-misinfo
            fact -1.989067    pro-true
          satire -1.233638    pro-true
           check -1.207405    pro-true
      fact check -1.189314    pro-true
       microsoft -0.977904    pro-true
   biden selling -0.815461    pro-true
           claim -0.757992    pro-true
         selling -0.756698    pro-true
  selling alaska -0.75669

In [60]:
log = BASE / "to-do/logs/day05_log.md"
log.write_text(f"""# Day05 Log — {pd.Timestamp.now()}
- Binary n: {len(df_bin)} (baseline {len(baseline)}, crisis {len(crisis)})
- LR F1 baseline {results['LR_baseline_test']['f1']:.3f} -> crisis {results['LR_crisis_test']['f1']:.3f} decay {decay_f1:.3f}
- RF F1 baseline {results['RF_baseline_test']['f1']:.3f} -> crisis {results['RF_crisis_test']['f1']:.3f}
- Mitigated (mix) crisis F1: {results.get('LR_mix_crisis_test', {}).get('f1', 'n/a')}
- Weights -> results/decision_boundary_weights.csv
- Metrics -> results/decay_metrics.json
""", encoding="utf-8")
print(f"Log -> {log}")


Log -> C:\Users\phoen\Code\Repos\jupyter\sm-bias\to-do\logs\day05_log.md
